# Candidate Clustering - Complete Pipeline Analysis

This notebook provides comprehensive visualization and analysis of the 3-phase clustering pipeline:
- **Phase 1**: Skill Normalization (1,271 → 786 skills)
- **Phase 2**: Feature Engineering (1,902 features)
- **Phase 3**: Multi-Label Clustering (12 clusters, distance-based)

In [ ]:
# Import libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print('Libraries loaded!')

## Phase 1: Skill Normalization Summary

In [ ]:
# Load normalization stats
with open('data/processed/normalization_stats_tier2.json', 'r') as f:
    norm_stats = json.load(f)

print('='*70)
print('PHASE 1: SKILL NORMALIZATION')
print('='*70)
print(f"Normalization Level: {norm_stats['normalization_level']}")
print(f"Total Candidates: {norm_stats['total_candidates']}")
print(f"\nSkill Reduction:")
print(f"  Before: {norm_stats['raw_skills_count']} unique skills")
print(f"  After:  {norm_stats['normalized_skills_count']} unique skills")
print(f"  Reduced by: {norm_stats['reduction_count']} skills ({norm_stats['reduction_percentage']:.1f}%)")
print(f"\nAvg Skills per Candidate:")
print(f"  Before: {norm_stats['avg_skills_per_candidate_before']:.1f}")
print(f"  After:  {norm_stats['avg_skills_per_candidate_after']:.1f}")

# Visualize top skills
top_skills = norm_stats['top_30_normalized_skills']
skills_df = pd.DataFrame(list(top_skills.items()), columns=['Skill', 'Count'])
skills_df = skills_df.sort_values('Count', ascending=True).tail(20)

plt.figure(figsize=(12, 8))
plt.barh(skills_df['Skill'], skills_df['Count'], color='steelblue')
plt.xlabel('Number of Candidates', fontsize=12)
plt.title('Top 20 Most Common Skills (After Normalization)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Phase 2: Feature Engineering Summary

In [ ]:
# Load Phase 2 stats
with open('data/processed/phase2_stats.json', 'r') as f:
    phase2_stats = json.load(f)

print('='*70)
print('PHASE 2: FEATURE ENGINEERING')
print('='*70)
print(f"Total Candidates: {phase2_stats['total_candidates']}")
print(f"Feature Matrix: {phase2_stats['feature_matrix_shape'][0]} × {phase2_stats['feature_matrix_shape'][1]}")
print(f"\nFeature Breakdown:")
for feature_type, count in phase2_stats['feature_breakdown'].items():
    print(f"  - {feature_type:30s}: {count:4d} features")

print(f"\nWeakness Modeling:")
print(f"  - Candidates with weaknesses: {phase2_stats['weakness_statistics']['candidates_with_weaknesses']}/{phase2_stats['total_candidates']}")
print(f"  - Avg weakness count: {phase2_stats['weakness_statistics']['avg_weakness_count']:.2f}")

# Visualize feature breakdown
feature_breakdown = phase2_stats['feature_breakdown']
plt.figure(figsize=(10, 6))
plt.bar(feature_breakdown.keys(), feature_breakdown.values(), color='coral')
plt.xlabel('Feature Type', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('Feature Engineering Breakdown', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Phase 3: Multi-Label Clustering Analysis

### Load Clustering Data

In [ ]:
# Load clustering results
with open('data/processed/clusters_final_multilabel.json', 'r', encoding='utf-8') as f:
    clusters_data = json.load(f)

# Load cluster assignments
with open('data/processed/cluster_assignments_multilabel.json', 'r') as f:
    assignments = json.load(f)

# Load normalized candidates
with open('data/processed/candidates_normalized_tier2.json', 'r', encoding='utf-8') as f:
    candidates = json.load(f)

# Load embeddings
embeddings = np.load('data/processed/embeddings_25d.npy')

print('='*70)
print('PHASE 3: MULTI-LABEL CLUSTERING')
print('='*70)
print(f"Clustering Method: {clusters_data['clustering_method']}")
print(f"Total Candidates: {clusters_data['total_candidates']}")
print(f"Total Clusters: {clusters_data['total_clusters']}")
print(f"\nMulti-Label Statistics:")
stats = clusters_data['multi_label_statistics']
print(f"  - Candidates in 1 cluster: {stats['candidates_in_1_cluster']}")
print(f"  - Candidates in 2+ clusters: {stats['candidates_in_2plus_clusters']}")
print(f"  - Avg clusters per candidate: {stats['avg_clusters_per_candidate']:.2f}")
print(f"  - Max clusters per candidate: {stats['max_clusters_per_candidate']}")

### Cluster Overview

In [ ]:
# Create cluster summary dataframe
cluster_summary = []
for cluster in clusters_data['clusters']:
    cluster_summary.append({
        'Cluster ID': cluster['cluster_id'],
        'Label': cluster['cluster_label'][:50] + '...' if len(cluster['cluster_label']) > 50 else cluster['cluster_label'],
        'Size': cluster['size'],
        'Primary': cluster['primary_members'],
        'Secondary': cluster['secondary_members'],
        'Profile': cluster['profile']['profile_type'],
        'Strengths': len(cluster['profile']['strengths']),
        'Weaknesses': len(cluster['profile']['weaknesses'])
    })

cluster_df = pd.DataFrame(cluster_summary)
print('\nCluster Summary:')
print(cluster_df.to_string(index=False))

### Visualization 1: Cluster Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Total members
axes[0].barh(cluster_df['Cluster ID'], cluster_df['Size'], color='steelblue')
axes[0].set_xlabel('Total Members', fontsize=12)
axes[0].set_title('Cluster Size (Total Members)', fontsize=14, fontweight='bold')
axes[0].invert_yaxis()

# Primary vs Secondary
x = np.arange(len(cluster_df))
width = 0.35
axes[1].barh(x - width/2, cluster_df['Primary'], width, label='Primary', color='darkgreen')
axes[1].barh(x + width/2, cluster_df['Secondary'], width, label='Secondary', color='lightgreen')
axes[1].set_yticks(x)
axes[1].set_yticklabels(cluster_df['Cluster ID'])
axes[1].set_xlabel('Number of Members', fontsize=12)
axes[1].set_title('Primary vs Secondary Members', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

### Visualization 2: Multi-Label Distribution

In [ ]:
# Count how many clusters each candidate belongs to
num_clusters_per_candidate = [a['num_clusters'] for a in assignments]
cluster_count_dist = Counter(num_clusters_per_candidate)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
x = sorted(cluster_count_dist.keys())
y = [cluster_count_dist[i] for i in x]
axes[0].bar(x, y, color='coral')
axes[0].set_xlabel('Number of Clusters per Candidate', fontsize=12)
axes[0].set_ylabel('Number of Candidates', fontsize=12)
axes[0].set_title('Multi-Label Distribution', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)

# Pie chart: 1 cluster vs 2+ clusters
labels = ['1 Cluster', '2+ Clusters']
sizes = [stats['candidates_in_1_cluster'], stats['candidates_in_2plus_clusters']]
colors = ['lightblue', 'lightcoral']
axes[1].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Single vs Multi-Label Candidates', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### Visualization 3: 2D UMAP Projection with Clusters

In [ ]:
# Reduce to 2D for visualization
import umap

print('Computing 2D UMAP projection for visualization...')
reducer_2d = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embeddings_2d = reducer_2d.fit_transform(embeddings)

# Create mapping of candidate_id to primary cluster
candidate_to_cluster = {a['candidate_id']: a['primary_cluster'] for a in assignments}

# Get primary cluster for each candidate
candidate_ids = [c['candidate_id'] for c in candidates]
primary_clusters = [candidate_to_cluster[cid] for cid in candidate_ids]

# Plot
plt.figure(figsize=(14, 10))
scatter = plt.scatter(
    embeddings_2d[:, 0],
    embeddings_2d[:, 1],
    c=primary_clusters,
    cmap='tab20',
    s=100,
    alpha=0.6,
    edgecolors='black',
    linewidth=0.5
)

plt.colorbar(scatter, label='Cluster ID')
plt.xlabel('UMAP Dimension 1', fontsize=12)
plt.ylabel('UMAP Dimension 2', fontsize=12)
plt.title('2D UMAP Projection of Candidates (Colored by Primary Cluster)', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('\n2D embeddings computed and saved for further analysis.')

### Visualization 4: Top Skills per Cluster

In [ ]:
# Show top 3 skills for each cluster
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

for i, cluster in enumerate(clusters_data['clusters']):
    ax = axes[i]
    
    # Get top 5 strengths
    strengths = cluster['profile']['strengths'][:5]
    if strengths:
        skills = [s['skill'] for s in strengths]
        scores = [s['avg_score'] for s in strengths]
        
        bars = ax.barh(skills, scores, color='green', alpha=0.7)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Avg Score', fontsize=10)
        ax.set_title(f"{cluster['cluster_id']} (n={cluster['size']})", fontsize=11, fontweight='bold')
        ax.invert_yaxis()
        
        # Add score labels
        for bar, score in zip(bars, scores):
            ax.text(score + 1, bar.get_y() + bar.get_height()/2, f'{score:.1f}',
                   va='center', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'No strengths', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f"{cluster['cluster_id']} (n={cluster['size']})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.suptitle('Top 5 Strengths per Cluster', fontsize=16, fontweight='bold', y=1.00)
plt.show()

### Visualization 5: Weaknesses per Cluster

In [ ]:
# Show weaknesses
fig, axes = plt.subplots(4, 3, figsize=(18, 16))
axes = axes.flatten()

for i, cluster in enumerate(clusters_data['clusters']):
    ax = axes[i]
    
    # Get top 5 weaknesses
    weaknesses = cluster['profile']['weaknesses'][:5]
    if weaknesses:
        skills = [w['skill'] for w in weaknesses]
        scores = [w['avg_score'] for w in weaknesses]
        
        bars = ax.barh(skills, scores, color='red', alpha=0.7)
        ax.set_xlim(0, 100)
        ax.set_xlabel('Avg Score', fontsize=10)
        ax.set_title(f"{cluster['cluster_id']} (n={cluster['size']})", fontsize=11, fontweight='bold')
        ax.invert_yaxis()
        
        # Add score labels
        for bar, score in zip(bars, scores):
            ax.text(score + 1, bar.get_y() + bar.get_height()/2, f'{score:.1f}',
                   va='center', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'No weaknesses', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(f"{cluster['cluster_id']} (n={cluster['size']})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.suptitle('Top 5 Weaknesses per Cluster', fontsize=16, fontweight='bold', y=1.00)
plt.show()

## Interactive Exploration: Explain Why a Candidate Belongs to Clusters

### Function to Explain Candidate Clustering

In [ ]:
def explain_candidate_clustering(candidate_id, top_k=5):
    """
    Explain why a candidate belongs to their assigned clusters.
    
    Parameters:
    - candidate_id: The ID of the candidate to explain
    - top_k: Number of top skills to show
    """
    # Find candidate data
    candidate = next((c for c in candidates if c['candidate_id'] == candidate_id), None)
    if not candidate:
        print(f"Candidate {candidate_id} not found!")
        return
    
    # Find assignment
    assignment = next((a for a in assignments if a['candidate_id'] == candidate_id), None)
    if not assignment:
        print(f"Assignment for {candidate_id} not found!")
        return
    
    # Get candidate index
    candidate_idx = next(i for i, c in enumerate(candidates) if c['candidate_id'] == candidate_id)
    
    print('='*80)
    print(f'CANDIDATE: {candidate_id}')
    print('='*80)
    
    # Show candidate's skills
    print(f"\nTotal Skills: {len(candidate['normalized_skills'])}")
    print(f"\nTop {top_k} Skills (by score):")
    sorted_skills = sorted(
        candidate['normalized_scores'].items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]
    for skill, score in sorted_skills:
        print(f"  - {skill:50s} | Score: {score:5.1f}")
    
    # Show cluster assignments
    print(f"\n{'='*80}")
    print(f"CLUSTER ASSIGNMENTS: {len(assignment['clusters'])} cluster(s)")
    print('='*80)
    
    for i, cluster_info in enumerate(assignment['clusters']):
        cluster_id = cluster_info['cluster_id']
        cluster_score = cluster_info['score']
        is_primary = (cluster_id == assignment['primary_cluster'])
        
        # Get cluster data
        cluster = next(c for c in clusters_data['clusters'] if c['cluster_id'] == f'cluster_{cluster_id}')
        
        print(f"\n{'#'*80}")
        print(f"Cluster {i+1}/{len(assignment['clusters'])}: cluster_{cluster_id} {'[PRIMARY]' if is_primary else '[SECONDARY]'}")
        print(f"{'#'*80}")
        print(f"Label: {cluster['cluster_label']}")
        print(f"Assignment Score: {cluster_score:.4f} (higher = closer to cluster center)")
        print(f"Distance: {cluster_info.get('distance', 'N/A')}")
        print(f"Profile Type: {cluster['profile']['profile_type']}")
        print(f"Cluster Size: {cluster['size']} members ({cluster['primary_members']} primary)")
        
        # Show cluster's top strengths and check if candidate has them
        print(f"\n{'*'*80}")
        print("WHY THIS CLUSTER? - Matching Strengths:")
        print('*'*80)
        cluster_strengths = cluster['profile']['strengths'][:5]
        if cluster_strengths:
            for strength in cluster_strengths:
                skill = strength['skill']
                cluster_avg = strength['avg_score']
                candidate_score = candidate['normalized_scores'].get(skill, 0)
                
                match_status = '✓' if candidate_score > 0 else '✗'
                print(f"  {match_status} {skill:40s} | Cluster Avg: {cluster_avg:5.1f} | Candidate: {candidate_score:5.1f}")
        else:
            print("  No strengths defined for this cluster")
        
        # Show cluster's weaknesses and check if candidate has them
        print(f"\n{'*'*80}")
        print("Matching Weaknesses:")
        print('*'*80)
        cluster_weaknesses = cluster['profile']['weaknesses'][:5]
        if cluster_weaknesses:
            for weakness in cluster_weaknesses:
                skill = weakness['skill']
                cluster_avg = weakness['avg_score']
                candidate_score = candidate['normalized_scores'].get(skill, 0)
                
                match_status = '✓' if candidate_score > 0 and candidate_score < 65 else '✗'
                print(f"  {match_status} {skill:40s} | Cluster Avg: {cluster_avg:5.1f} | Candidate: {candidate_score:5.1f}")
        else:
            print("  No weaknesses defined for this cluster")
    
    # Visualize candidate position in 2D space
    print(f"\n{'='*80}")
    print("VISUAL EXPLANATION: Candidate Position in 2D Space")
    print('='*80)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    # Plot all candidates (gray)
    ax.scatter(
        embeddings_2d[:, 0],
        embeddings_2d[:, 1],
        c='lightgray',
        s=50,
        alpha=0.3,
        edgecolors='none'
    )
    
    # Highlight clusters this candidate belongs to
    colors = plt.cm.tab10(np.linspace(0, 1, len(assignment['clusters'])))
    for i, cluster_info in enumerate(assignment['clusters']):
        cluster_id = cluster_info['cluster_id']
        
        # Get all candidates in this cluster
        cluster_candidate_ids = next(c['members'] for c in clusters_data['clusters'] if c['cluster_id'] == f'cluster_{cluster_id}')
        cluster_indices = [j for j, c in enumerate(candidates) if c['candidate_id'] in cluster_candidate_ids]
        
        ax.scatter(
            embeddings_2d[cluster_indices, 0],
            embeddings_2d[cluster_indices, 1],
            c=[colors[i]],
            s=100,
            alpha=0.5,
            label=f"cluster_{cluster_id}",
            edgecolors='black',
            linewidth=0.5
        )
    
    # Highlight the target candidate
    ax.scatter(
        embeddings_2d[candidate_idx, 0],
        embeddings_2d[candidate_idx, 1],
        c='red',
        s=500,
        marker='*',
        edgecolors='black',
        linewidth=2,
        label=f'TARGET: {candidate_id[:8]}...',
        zorder=100
    )
    
    ax.set_xlabel('UMAP Dimension 1', fontsize=12)
    ax.set_ylabel('UMAP Dimension 2', fontsize=12)
    ax.set_title(f'Candidate {candidate_id[:8]}... Position in 2D Space', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print('Function defined! Use: explain_candidate_clustering(candidate_id)')

### Example 1: Explain a Random Candidate

In [ ]:
# Pick a random candidate with multiple clusters
multi_cluster_candidates = [a for a in assignments if a['num_clusters'] > 2]
sample_candidate = multi_cluster_candidates[0]['candidate_id']

explain_candidate_clustering(sample_candidate)

### Example 2: Explain a Candidate with Embedding Skills

In [ ]:
# Find a candidate with embedding skills
embedding_candidate = None
for candidate in candidates:
    if any('embedding' in skill.lower() for skill in candidate['normalized_skills']):
        embedding_candidate = candidate['candidate_id']
        break

if embedding_candidate:
    explain_candidate_clustering(embedding_candidate)
else:
    print('No candidate with embedding skills found')

### Example 3: Explain a Candidate in AI/ML Cluster

In [ ]:
# Find a candidate in cluster_8 (Machine Learning cluster)
ml_cluster = next(c for c in clusters_data['clusters'] if c['cluster_id'] == 'cluster_8')
ml_candidate = ml_cluster['members'][0]

explain_candidate_clustering(ml_candidate)

### Interactive: Explore Any Candidate

Replace `YOUR_CANDIDATE_ID` with any candidate ID from the dataset.

In [ ]:
# List some candidate IDs to choose from
print('Sample Candidate IDs:')
for i, candidate in enumerate(candidates[:10]):
    assignment = next(a for a in assignments if a['candidate_id'] == candidate['candidate_id'])
    print(f"{i+1}. {candidate['candidate_id']} (in {assignment['num_clusters']} cluster(s))")

print('\nUse: explain_candidate_clustering("<candidate_id>")')

In [ ]:
# YOUR CUSTOM EXPLORATION HERE
# explain_candidate_clustering('YOUR_CANDIDATE_ID')

## Summary Statistics

In [ ]:
print('='*80)
print('COMPLETE PIPELINE SUMMARY')
print('='*80)

print(f"\nPhase 1 - Skill Normalization:")
print(f"  - Raw skills: {norm_stats['raw_skills_count']}")
print(f"  - Normalized skills: {norm_stats['normalized_skills_count']}")
print(f"  - Reduction: {norm_stats['reduction_percentage']:.1f}%")

print(f"\nPhase 2 - Feature Engineering:")
print(f"  - Feature matrix: {phase2_stats['feature_matrix_shape'][0]} × {phase2_stats['feature_matrix_shape'][1]}")
print(f"  - Total features: {phase2_stats['total_features']}")

print(f"\nPhase 3 - Multi-Label Clustering:")
print(f"  - Total clusters: {clusters_data['total_clusters']}")
print(f"  - Clustering method: {clusters_data['clustering_method']}")
print(f"  - Avg clusters per candidate: {stats['avg_clusters_per_candidate']:.2f}")
print(f"  - Multi-label candidates: {stats['candidates_in_2plus_clusters']}/{clusters_data['total_candidates']} ({stats['candidates_in_2plus_clusters']/clusters_data['total_candidates']*100:.1f}%)")

print('\n' + '='*80)
print('ANALYSIS COMPLETE!')
print('='*80)